# Урок 5 · Гістограма — «паспорт» зображення
**Як бачить машина · 9-А · робочий зошит**

**Спершу:** *Файл → Зберегти копію на Диску*. Без цього ваші зміни зникнуть після уроку.

**Як працювати**
* Кроки йдуть так само, як на слайдах: слайд «Зошит · крок 2» = розділ «Крок 2» тут.
* ▶ ліворуч від клітинки — запустити її. Клітинки запускаємо **згори вниз**.
* `___` — пропуск: його заповнюєте ви.
* 💡 **Підказка** — клітинка з прихованим кодом. Відкривайте, лише якщо застрягли.
* Своє фото: панель 📁 ліворуч → завантажити файл → назвати **foto.jpg**. Немає фото — зошит візьме тестове.

In [ ]:
#@title Службове: тестове фото й малювання (можна не відкривати) { display-mode: "form" }
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def test_photo():
    """Тестове фото 640×480: небо, сонце й пагорби. Беремо, якщо свого немає."""
    h, w = 480, 640
    y, x = np.mgrid[0:h, 0:w]
    a = 200 - y * 0.25                              # небо: світліше вгорі
    a[(x - 150)**2 + (y - 140)**2 < 60**2] = 235    # сонце
    hill = 330 + 40 * np.sin(x / 90)
    a = np.where(y > hill, 90 - (y - 300) * 0.1, a) # пагорби
    return np.clip(a, 0, 255).astype(np.uint8)

def show(*imgs, titles=None):
    """Показати кілька сірих фото поруч."""
    fig, ax = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 3))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(im, cmap='gray', vmin=0, vmax=255)
        ax[i].axis('off')
        if titles: ax[i].set_title(titles[i])
    plt.show()

def bars(*imgs, titles=None):
    """Гістограми кількох фото поруч."""
    fig, ax = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 2.6))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        h, _ = np.histogram(im, bins=256, range=(0, 256))
        ax[i].bar(range(256), h, width=1)
        ax[i].set_xlim(0, 255); ax[i].set_yticks([])
        if titles: ax[i].set_title(titles[i])
    plt.show()

## Крок 0 · Беремо фото
Запустіть клітинку. Вона надрукує розмір фото, найтемніший і найсвітліший піксель.

In [ ]:
if os.path.exists('foto.jpg'):
    a = np.array(Image.open('foto.jpg').convert('L'))
    print('Беремо ваше фото foto.jpg')
else:
    a = test_photo()
    print('foto.jpg не знайдено — беремо тестове фото')
print(a.shape, a.min(), a.max())
show(a)

## Крок 1 · Гістограма руками
Ось таблиця 4 × 4 — крихітне «фото».

| 30 | 30 | 90 | 30 |
|---|---|---|---|
| 30 | 90 | 150 | 30 |
| 90 | 30 | 30 | 210 |
| 30 | 30 | 90 | 30 |

1. **На папері:** скільки разів трапляється кожне число? Намалюйте гістограму.
2. Запустіть клітинку й перевірте себе: перший рядок — які числа є, другий — скільки разів кожне.
3. Темне це фото чи світле?

In [ ]:
t = np.array([[30, 30,  90,  30],
              [30, 90, 150,  30],
              [90, 30,  30, 210],
              [30, 30,  90,  30]])

v, c = np.unique(t, return_counts=True)   # які числа є і скільки разів кожне
print(v)
print(c)
plt.bar(v, c, width=12); plt.xlim(0, 255); plt.show()

## Крок 2 · Гістограма свого фото
`np.histogram` розкладає всі пікселі по 256 «кошиках» — по одному на кожну яскравість.
`h[k]` — скільки пікселів мають яскравість `k`.

In [ ]:
h, _ = np.histogram(a, bins=256, range=(0, 256))   # h[k] — скільки пікселів яскравості k
plt.bar(range(256), h, width=1)
plt.xlim(0, 255)
plt.show()
print(h[235])        # скільки пікселів мають яскравість 235

✍️ **Ваша відповідь:** де на гістограмі темні ділянки вашого фото, а де світлі? Є вузькі піки — що це на фото?

*(пишіть тут)*

## Крок 3 · Чия це гістограма?
Клітинка робить із вашого фото три нові: **A**, **B**, **C** — і малює їхні гістограми **в іншому порядку**: 1, 2, 3.
Знайдіть пари.

In [ ]:
dark   = (a * 0.4).astype(np.uint8)                  # A
flat   = (100 + a * 0.25).astype(np.uint8)           # B
bright = (130 + a * 0.5).astype(np.uint8)            # C
show(dark, flat, bright, titles=['A', 'B', 'C'])
bars(bright, dark, flat, titles=['1', '2', '3'])

✍️ **Пари:** A — …, B — …, C — …

*(пишіть тут)*

## Крок 4 · Розтягуємо контраст
Фото **B** сіре й «пласке»: усі пікселі стиснулись у вузьку купку. Розтягнемо її на всю шкалу 0…255:

**нове = (старе − min) · 255 / (max − min)**

Заповніть два пропуски `___` і запустіть.

In [ ]:
f = flat                   # сіре «пласке» фото B з кроку 3
lo = f.___()               # найтемніший піксель
hi = f.___()               # найсвітліший піксель
s = ((f.astype(int) - lo) * 255 / (hi - lo)).astype(np.uint8)
print(lo, hi, '→', s.min(), s.max())
show(f, s, titles=['до', 'після'])
bars(f, s, titles=['до', 'після'])

In [ ]:
#@title 💡 Підказка { display-mode: "form" }
# Найменше число в таблиці — f.min(), найбільше — f.max()
lo = f.min()
hi = f.max()

**Встигли?** Розтягніть своє фото `a` (замініть `f = flat` на `f = a`). Чому воно змінилося зовсім мало?

## Крок 5 · Автоматика «фото темне?»
`a < 85` — таблиця з `True` / `False`. `.sum()` рахує `True` — тобто пікселі, темніші за 85.
Замініть `___` на частку, після якої фото вважаємо темним: «темних більше половини».

In [ ]:
dark_px = (a < 85).sum()           # скільки пікселів темніші за 85
share = dark_px / a.size           # яка це частка від усіх
print(dark_px, 'з', a.size, '→', round(share * 100), '%')
if share > ___:
    print('Фото темне — варто освітлити')
else:
    print('Освітлення нормальне')

In [ ]:
#@title 💡 Підказка { display-mode: "form" }
# «Більше половини» — це частка 0.5
# if share > 0.5:

**Встигли?** Замініть у першому рядку `a` на `dark` — фото A з кроку 3. Що надрукує тепер?

## ДЗ · Три гістограми без фото
Запустіть клітинку — вона намалює гістограми трьох фото, яких ви не бачите.

In [ ]:
#@title ДЗ: три загадкові гістограми { display-mode: "form" }
g = test_photo()
m1 = np.clip(255 - (255 - g) * 0.35, 0, 255).astype(np.uint8)
m2 = (g * 0.3 + 20).astype(np.uint8)
m3 = np.clip((g.astype(int) - 150) * 1.6 + 128, 0, 255).astype(np.uint8)
bars(m1, m2, m3, titles=['1', '2', '3'])

✍️ **Відповідь для кожної гістограми:** темне чи світле фото, контрастне чи ні — і **чому** (де купка, яка вона широка).

1. …
2. …
3. …